<a id="top"></a>
# Manual testing of synced selection and filtering between `mast-table` and `mast-aladin` 

This notebook walks through a series of API calls and UI interactions to confirm functionality of syncing selections and filters between `mast-table` and `mast-aladin` . Run cells in order, confirm expected behavior as cells are executed.

Testing includes
- selections and filters applied in `mast-table` are reflected in `mast-aladin`
- selections applied in `mast-aladin` are reflected in `mast-table`
- multiple table overlays maintain expected functionality of synced selection and filtering
- users can update overlays in `mast-aladin` via API calls
- users can remove overlays in `mast-aladin` via API calls


## Setup
Import required packages and generate two distinct example tables (Roman and HST) to avoid query times.


In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

from mast_aladin import MastAladin
from mast_table import MastTable
from mast_aladin.utils.footprint_generator import (
   Exposure, CustomPattern, DitherPattern, exp_list_to_table
)

target = SkyCoord(ra=202.4696*u.deg, dec=47.1953*u.deg, frame="icrs")

In [ ]:
# generate sample Roman observation table
roman_exp = Exposure(target.ra, target.dec)
roman_pattern = DitherPattern(
    roman_exp,
    num_rows=1, num_cols=1,
    row_x_offset=300, row_y_offset=10,
    col_x_offset=0, col_y_offset=130
)

roman_obs = exp_list_to_table(roman_pattern.get_exp_list(pa=90))

In [ ]:
# generate custom pattern that looks like a 3x2 grid with HST WFC3 UVIS
hst_exp = Exposure(
    target.ra, target.dec,
   telescope="HST", 
   instrument="WFC3", 
   aperture="IUVIS"
)
hst_pattern = CustomPattern(hst_exp,[
    [0, 0],
    [150, 0],
    [300, 0],
    [0, 150],
    [150, 150],
    [300, 150],
])

hst_obs = exp_list_to_table(hst_pattern.get_exp_list())

## Create and display `mast-aladin`

In [ ]:
mast_aladin = MastAladin(
    fov=3,
    target=target,
    show_selection_mode_control=True,
    selection_mode="skewer",
    full_screen=True
)

# defaults to display in a sidecar split to the right
mast_aladin.show()

## Create and display `mast-table` instances

In [ ]:
# for generated Roman pattern
roman_table = MastTable(roman_obs)
roman_table_id = id(roman_table)
roman_table.show(loc="sidecar:split-bottom", ref = mast_aladin, title = "roman")

# for HST table
hst_table = MastTable(hst_obs)
hst_table_id = id(hst_table)
hst_table.show(loc="sidecar:tab-after", ref = roman_table, title = "hst")

## Add first `mast-table` of Roman data as an overlay in `mast-aladin`

In [ ]:
roman_overlay = mast_aladin.add_table(
    roman_table,
    color="#FF6600",
)

<div class="alert alert-block alert-info">
    Run the following cell and confirm we get no errors.
</div>

In [ ]:
# confirm the number of listeners and that removed overlays are handled properly
assert list(mast_aladin._mast_table_listeners.keys()) == [
    roman_table_id
]
assert list(mast_aladin._mast_table_names.keys()) == [
    "catalog"
]
assert list(mast_aladin._selected_mask_listeners.keys()) == [
    "catalog"
]

<div class="alert alert-block alert-info">
    Perform the following tests for the Roman overlay:
    
    1. Set a selection using mast-table UI. Confirm selections are also highlighted in green in mast-aladin window.
    2. Select a footprint using mast-aladin UI. Confirm selection is also set in mast-table.
    3. Set selection using mast-table API (via first cell immediately below this). Confirm selections are highlighted in green in mast-aladin window and differ from step 1 above.
    4. Set filter for `aper_dec >= 47.19598169497287`. Confirm that only 6 footprints are now visible in mast-aladin. Confirm that there are now only 3 selections remaining (in the table AND in mast-aladin). 
    5. Update the overlay using the second cell below to change the color and name. Confirm name is updated in aladin stack menu (to "Roman Observations") and the color is now showing as pink ("#FF69B4"). Confirm existing selections and filter are still applied to the overlay in mast-aladin. Confirm applying a new selection and removing existing filter is still possible and synced between the widgets.
</div>

In [ ]:
roman_table.select(slice(0,6))

In [ ]:
roman_overlay = roman_overlay.update(
    name="Roman Observations",
    color="#FF69B4"
)

## Add second `mast-table` of HST data as an overlay in `mast-aladin`

In [ ]:
hst_overlay= mast_aladin.add_table(
    hst_table, color="#FF6600",
)

<div class="alert alert-block alert-info">
    Run the following cell and confirm we get no errors.
</div>

In [ ]:
# confirm the number of listeners and that updated overlay names are tracked properly
assert list(mast_aladin._mast_table_listeners.keys()) == [
    roman_table_id,hst_table_id
]
assert list(mast_aladin._mast_table_names.keys()) == [
    "Roman Observations","catalog"
]
assert list(mast_aladin._selected_mask_listeners.keys()) == [
    "Roman Observations","catalog"
]

<div class="alert alert-block alert-info">
    Perform the following tests for the HST overlay:
    
    1. Confirm the name shown in the mast-aladin stack is "catalog".
    2. Set a selection using mast-table UI. Confirm selections are also highlighted in green in mast-aladin window.
    3. Select a footprint using mast-aladin UI. Confirm selection is also set in mast-table. If your pick also sets a selection in the Roman layer, confirm its selection is also properly set in mast-table.
    4. Set selection using mast-table API (via first cell immediately below this). Confirm selections are highlighted in green in mast-aladin window and differ from step 1 above. Confirm no changes are seen in the Roman layer.
    5. Set filter for `x_off == 300`. Confirm that only 2 number of footprints are now visible in mast-aladin. Confirm that there is now only 1 selection remaining (in the table AND in mast-aladin). Confirm no changes are seen in the Roman layer.
    6. Update the overlay using the second cell below to change the color and name. Confirm name is updated in aladin stack menu (to "HST Observations") and the color is now showing as blue ("#0000FF"). Confirm existing selections and filters of your choice are still applied to the overlay.
</div>

In [ ]:
hst_table.select([0,2,4])

In [ ]:
hst_overlay = hst_overlay.update(
    name = "HST Observations",
    color = "#0000FF",
)

<div class="alert alert-block alert-info">
    Run the cell below and confirm we see no errors or warnings.
</div>

In [ ]:
# confirm the number of listeners and that updated overlay names are tracked properly
assert list(mast_aladin._mast_table_listeners.keys()) == [
    roman_table_id,hst_table_id
]
assert list(mast_aladin._mast_table_names.keys()) == [
    "Roman Observations","HST Observations"
]
assert list(mast_aladin._selected_mask_listeners.keys()) == [
    "Roman Observations","HST Observations"
]

## Remove overlay

<div class="alert alert-block alert-info">
    Run the cell below and confirm we no longer see the HST overlay in mast-aladin (both visually and in the stack menu).
</div>

In [ ]:
mast_aladin.remove_overlay(roman_overlay)

<div class="alert alert-block alert-info">
    Run the following cell and confirm we get no errors.
</div>

In [ ]:
# confirm the number of listeners and that removed overlays are handled properly
assert list(mast_aladin._mast_table_listeners.keys()) == [
    hst_table_id
]
assert list(mast_aladin._mast_table_names.keys()) == [
    "HST Observations"
]
assert list(mast_aladin._selected_mask_listeners.keys()) == [
    "HST Observations"
]

## About this notebook

**Last updated:** November 2026
***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 